# Notebook 2 of 4 — Feature Engineering and Split

Input: `data/Churn_Modelling.csv`.  Output: `data/splits.pkl` (train/test split + preprocessor), `data/Cleaned_data.csv`.  Next: `03_Model_Training.ipynb`.

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "model"


## 12. Feature engineering — only what EDA earned

I do not add features to make the pipeline look larger. Each one is a pattern the tables already showed, written in a form a model can use:

| Feature | Why it exists |
|---|---|
| `Zero_Balance` | Empty accounts churn at 13.8%, funded accounts at 24.1% |
| `Balance_Salary_Ratio` | Salary itself is flat; the ratio asks whether the deposit is large relative to income |
| `Products_Bucket` | Encodes the cliff: 7.6% at 2 products versus 83–100% at 3–4 |
| `Age_Inactive` | The two strongest signals, together |
| `Germany_HighBal` | The market where rate and balance are both worst |

**Preprocessing, and why it is inside the pipeline**

- Scale numeric fields so distance-based models are not dominated by balance (tens of thousands) over age (tens).
- Encode `Geography` and `Gender` so the model can use them without me hand-building every dummy in the app.
- **Stratified 80/20 split** so the test set keeps the same 20% churn rate as the book. A random split can accidentally make the test set look easier.
- Resampling happens **after** the split and only on training folds. Touching the test set would inflate the score the business will be shown.

In [ ]:
df = pd.read_csv(DATA_DIR / "Churn_Modelling.csv")

df["Zero_Balance"]         = (df["Balance"] == 0).astype(int)
df["Balance_Salary_Ratio"] = df["Balance"] / (df["EstimatedSalary"] + 1)
df["Products_Bucket"]      = (df["NumOfProducts"] >= 3).astype(int)   # 2-vs-3+ cliff
df["Age_Inactive"]         = df["Age"] * (df["IsActiveMember"] == 0)  # older + inactive
df["Germany_HighBal"]      = ((df["Geography"] == "Germany") &
                               (df["Balance"] > df["Balance"].median())).astype(int)

num_features = ["Age", "Balance", "CreditScore", "EstimatedSalary",
                "NumOfProducts", "IsActiveMember",
                "Zero_Balance", "Balance_Salary_Ratio",
                "Products_Bucket", "Age_Inactive", "Germany_HighBal"]
cat_features = ["Geography", "Gender"]

X = df[num_features + cat_features]
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OrdinalEncoder(), cat_features)
])

print(X_train.shape, X_test.shape)

### Save the split for the next notebook

In [ ]:
with open(DATA_DIR / "splits.pkl", "wb") as f:
    pickle.dump((X_train, X_test, y_train, y_test, preprocessor), f)

### Save cleaned data

In [ ]:
num_features = ['Age', 'Balance', 'CreditScore', "EstimatedSalary","NumOfProducts","IsActiveMember"]
cat_features = ['Geography', 'Gender']
X  = num_features + cat_features 
X.append("Exited")
df = df[X]
df.to_csv(DATA_DIR / "Cleaned_data.csv", index=False)